In [25]:
import sys
!{sys.executable} -m pip install -q yfinance pandas-datareader scikit-learn xgboost lightgbm shap xlsxwriter


In [26]:
import sys
!{sys.executable} -m pip install -q imblearn

In [27]:
from imblearn.over_sampling import SMOTE

In [28]:
import os, re, time, json, warnings
from pathlib import Path
from typing import Dict, List

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import yfinance as yf
import pandas_datareader as pdr
import shap

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTE

# =============================================================================
# CONFIG
# =============================================================================
#
# OBJECTIF :
#   1. Sélection SHAP (XGBoost pilote) des meilleures features de base
#   2. Génération d'interactions COMPLÈTES (simples + complexes)
#   3. Re-sélection SHAP sur (base + interactions) → top 30 final
#   4. TRAIN_START déterminé automatiquement par la couverture des features
#   5. Split fixe train < 2022 / test 2022→2026
#   6. 5 algos × N=5-15 features × 4 régimes × 3 horizons

OUTPUT_DIR = Path("/content/outputs_v23_shap_interactions_final")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE  = 42
MIN_DATA_START = pd.Timestamp("2000-01-01")
TRAIN_START    = "2000-01-01"   # pour le téléchargement initial uniquement
TEST_DATE      = "2022-01-01"   # split fixe : train < 2022 / test 2022→2026

YF_CHUNK_SIZE         = 40
SLEEP_BETWEEN_CHUNKS  = 1.0
MIN_COLUMN_COVERAGE   = 0.90
MAX_FIRST_VALID_LAG_DAYS = 365
VIX_FLAT_PCT_THRESHOLD   = 0.0
ROLLING_QUANTILE_WINDOW  = 504

# --- Phase 1 : sélection SHAP ---
SHAP_PILOT_N_ESTIMATORS = 150   # XGBoost pilote (rapide)
SHAP_TOP_BASE_N  = 40   # features de base à garder avant génération d'interactions
SHAP_TOP_FINAL_N = 30   # features finales (base + interactions) après 2e passe SHAP

# --- Phase 1 bis : interactions complexes ---
INTERACTION_ROLLING_WINDOW = 20   # fenêtre pour MA croisées et z-scores relatifs
N_TOP_FOR_INTERACTIONS = 20       # paires calculées sur le top-N uniquement
                                   # (limiter le volume : 20×19/2 = 190 paires)

# --- Phase 2 : entraînement ---
HORIZONS_TO_TEST      = [1, 3, 5]
ALL_REGIMES           = ["CALM", "NORMAL", "STRESS", "GLOBAL"]
ALL_ALGOS             = ["XGBoost", "LightGBM", "GradientBoosting",
                         "RandomForest", "LogisticRegression"]
MIN_N_FEATURES        = 5
MAX_N_FEATURES        = 15

np.random.seed(RANDOM_STATE)

FRED_API_KEY = os.getenv("FRED_API_KEY")
if FRED_API_KEY:
    os.environ["FRED_API_KEY"] = FRED_API_KEY


In [29]:
# (select_and_filter_features est definie dans la cellule suivante - cette
# cellule etait un doublon residuel, neutralisee pour eviter la confusion.)


In [30]:
# =============================================================================
# Sélection de features par corrélation (reprise à l'identique du notebook
# original) : produit une LISTE ORDONNÉE de features (les plus corrélées au
# target en premier, en excluant les features trop corrélées entre elles).
# Cette liste classée sert ensuite de base pour tester N=1..20 features.
# =============================================================================

correlation_threshold_target = 0.05
correlation_threshold_features = 0.9

def select_and_filter_features(
    df: pd.DataFrame,
    all_features: List[str],
    target_col: str = "VIX_Direction",
    correlation_threshold_target: float = 0.05,
    correlation_threshold_features: float = 0.9,
    max_features_to_select: int = 20
) -> List[str]:
    cols_to_use = [f for f in all_features if f in df.columns] + [target_col]
    df_for_corr = df[cols_to_use].copy().dropna()

    if df_for_corr.empty:
        warnings.warn("DataFrame for correlation is empty after dropping NaNs.")
        return []

    correlations = df_for_corr.corr(method='spearman')[target_col].abs().sort_values(ascending=False)

    selected_features_initial = correlations[correlations >= correlation_threshold_target].index.tolist()
    if target_col in selected_features_initial:
        selected_features_initial.remove(target_col)

    if not selected_features_initial:
        return []

    features_small = []
    high_corr_features = list(selected_features_initial)
    df_for_inter_corr = df_for_corr[[f for f in high_corr_features if f in df_for_corr.columns]].copy()

    while len(high_corr_features) > 0 and len(features_small) < max_features_to_select:
        current_correlations = correlations[high_corr_features]
        if current_correlations.empty:
            break
        f = current_correlations.idxmax()
        features_small.append(f)
        high_corr_features.remove(f)

        correlated_with_f = []
        if f in df_for_inter_corr.columns:
            for f_other in high_corr_features:
                if f_other in df_for_inter_corr.columns:
                    try:
                        corr_value = df_for_inter_corr[[f, f_other]].corr(method='spearman').iloc[0, 1]
                        if abs(corr_value) >= correlation_threshold_features:
                            correlated_with_f.append(f_other)
                    except Exception:
                        pass

        for f_to_remove in correlated_with_f:
            if f_to_remove in high_corr_features:
                high_corr_features.remove(f_to_remove)

    return features_small


In [31]:
# =============================================================================
# RECOMMANDATION 2 : génération systématique d'interactions explicites.
#
# Pour un ensemble de features de base (typiquement le top 30 actuel d'un
# régime/horizon), génère TOUTES les paires (ratio + différence) :
#   - ratio      : feat_i / feat_j  (nommé "{i}_div_{j}")
#   - différence : feat_i - feat_j  (nommé "{i}_minus_{j}")
#
# Pour 30 features : 30*29/2 = 435 paires non-ordonnées x 2 opérations = 870
# nouvelles colonnes potentielles. Toutes ne seront pas utiles - elles sont
# ensuite repassées dans select_and_filter_features (même filtre Spearman +
# anti-redondance) pour ne garder que celles qui apportent un signal réel.
#
# Protection : ratio i/j est mis à NaN si |j| < epsilon (évite divisions
# explosives qui pollueraient la sélection avec du bruit numérique).
# =============================================================================

def generate_pairwise_interactions(df: pd.DataFrame, base_features: list,
                                    epsilon: float = 1e-8) -> pd.DataFrame:
    """Génère ratio et différence pour toutes les paires non-ordonnées de
    base_features. Retourne un NOUVEAU dataframe (les colonnes générées
    uniquement, même index que df) - à concaténer par l'appelant."""
    n = len(base_features)
    print(f"[INTERACTIONS] Génération de paires pour {n} features de base "
          f"({n*(n-1)//2} paires x 2 opérations = {n*(n-1)} colonnes max)...")

    new_cols = {}
    for i in range(n):
        for j in range(i + 1, n):
            fi, fj = base_features[i], base_features[j]
            if fi not in df.columns or fj not in df.columns:
                continue

            col_i = df[fi]
            col_j = df[fj]

            # Différence
            diff_name = f"{fi}_minus_{fj}"
            new_cols[diff_name] = col_i - col_j

            # Ratio (protégé contre division par ~0)
            ratio_name = f"{fi}_div_{fj}"
            safe_denom = col_j.where(col_j.abs() >= epsilon, np.nan)
            new_cols[ratio_name] = col_i / safe_denom

    interactions_df = pd.DataFrame(new_cols, index=df.index)
    interactions_df = interactions_df.replace([np.inf, -np.inf], np.nan)
    print(f"[INTERACTIONS] {interactions_df.shape[1]} colonnes d'interactions générées.")
    return interactions_df


In [32]:
BAD_TICKERS = {
    "XXIV",
    "TVIX",
    "ZIV",
    "^MIB",
    "CELG",
    "AET",
    "HES",
    "GPS",
    "JWN",
    "DFS",
    "SPX",
    "EON",
    "EDF",
    "RWE",
    "SZR",
    "ICN",
    "CEIX",
    "MXEA",
    "SQ",
    "SHELL",
    "K",
}

MANUAL_YF_NAMES = {
    "^GSPC": "SP500_Price",
    "^IXIC": "NASDAQ_Price",
    "^DJI": "DOW_Price",
    "^RUT": "Russell_Price",
    "^VIX": "VIX_Price",
    "^VXN": "VXN_NASDAQ_Vol",
    "^OVX": "OVX_Oil_Vol",
    "^GVZ": "GVZ_Gold_Vol",
    "^EVZ": "EVZ_EUR_Vol",
    "^FTSE": "FTSE_UK",
    "^N225": "Nikkei_Japan",
    "^HSI": "HangSeng_HK",
    "^GDAXI": "DAX_Germany",
    "^FCHI": "CAC40_France",
    "^STOXX50E": "STOXX50E_EU",
    "SPY": "SPY",
    "QQQ": "QQQ",
    "TLT": "TLT_LongBond",
    "GLD": "GLD_Gold",
    "USO": "USO_Oil",
    "UUP": "UUP_Dollar",
    "FXE": "FXE_Euro",
    "FXY": "FXY_Yen",
    "HYG": "HYG_HighYield",
    "LQD": "LQD_InvGrade",

    # --- Tickers ajoutés (issus du dictionnaire massif fourni) ---
    "^BVSP": "BOVESPA_Brazil",
    "^AXJO": "ASX_Australia",
    "^AORD": "AORD_AUS",
    "^IBEX": "IBEX_Spain",
    "VXX": "VXX",
    "UVXY": "UVXY",
    "VIXY": "VIXY",
    "SVXY": "SVXY",
    "VXZ": "VXZ",
    "VIXM": "VIXM",
    "XLK": "XLK_Tech",
    "XLF": "XLF_Fin",
    "XLE": "XLE_Energy",
    "XLV": "XLV_Health",
    "XLU": "XLU_Util",
    "XLP": "XLP_Staples",
    "XLI": "XLI_Indust",
    "XLY": "XLY_Disc",
    "XLRE": "XLRE_RE",
    "XLB": "XLB_Materials",
    "XLC": "XLC_CommServ",
    "GOOGL": "GOOGL_Google",
    "META": "META_Meta",
    "AVGO": "AVGO_Broadcom",
    "ASML": "ASML_ASML",
    "WFC": "WFC_WellsFargo",
    "GS": "GS_GoldmanSachs",
    "BLK": "BLK_BlackRock",
    "SCHW": "SCHW_Schwab",
    "MS": "MS_MorganStanley",
    "COF": "COF_CapitalOne",
    "BAX": "BAX_BankBoston",
    "AXP": "AXP_Amex",
    "EQR": "EQR_Equity",
    "PLD": "PLD_Prologis",
    "AMT": "AMT_AmericanTower",
    "EQIX": "EQIX_Equinix",
    "CCI": "CCI_CrownCastle",
    "PSA": "PSA_PublicStorage",
    "ABBV": "ABBV_AbbVie",
    "MRK": "MRK_Merck",
    "BMY": "BMY_BristolMyers",
    "AMGN": "AMGN_Amgen",
    "GILD": "GILD_Gilead",
    "BNTX": "BNTX_BioNTech",
    "MRNA": "MRNA_Moderna",
    "CRSP": "CRSP_CrisprTherapy",
    "VRTX": "VRTX_VertexPharm",
    "ILMN": "ILMN_Illumina",
    "DXCM": "DXCM_Dexcom",
    "TDOC": "TDOC_Teladoc",
    "CI": "CI_Cigna",
    "HUM": "HUM_Humana",
    "RTX": "RTX_Raytheon",
    "LMT": "LMT_LockheedMartin",
    "NOC": "NOC_Northrop",
    "GD": "GD_GeneralDynamics",
    "CAT": "CAT_Caterpillar",
    "DE": "DE_Deere",
    "ITT": "ITT_ITTInc",
    "PAYX": "PAYX_Paychex",
    "CTAS": "CTAS_Cintas",
    "MMM": "3M",
    "HON": "HON_Honeywell",
    "ETN": "ETN_Eaton",
    "EMR": "EMR_Emerson",
    "OTIS": "OTIS_Otis",
    "JCI": "JCI_JohnsonControls",
    "PTC": "PTC_PTC",
    "SMCI": "SMCI_SuperMicroComputer",
    "COP": "COP_ConocoPhillips",
    "SLB": "SLB_Schlumberger",
    "EOG": "EOG_EOGResources",
    "MPC": "MPC_MarathonPetroleum",
    "PSX": "PSX_PhillipsLiquids",
    "VLO": "VLO_Valero",
    "PM": "PM_PhilipMorris",
    "MO": "MO_AltriaMG",
    "BTI": "BTI_BritishAmerican",
    "TBP": "TBP_Tata",
    "BP": "BP_BritishPetroleum",
    "TTE": "TTE_TotalEnergies",
    "ENB": "ENB_EnbridgeInc",
    "MET": "MET_MetalexEnergy",
    "ADM": "ADM_ArcherDaniels",
    "MKC": "MKC_McCormick",
    "SJM": "SJM_JM_Smucker",
    "CPB": "CPB_CampbellSoup",
    "GIS": "GIS_GeneralMills",
    "MDLZ": "MDLZ_Mondelez",
    "NSRGY": "NSRGY_Nestle",
    "TAP": "TAP_MolsonCoors",
    "BDX": "BDX_Becton_Dickinson",
    "CLX": "CLX_Clorox",
    "CL": "CL_Colgate",
    "UL": "UL_Unilever",
    "LVRK": "LVRK_Lavazza",
    "YUM": "YUM_YumBrands",
    "QSR": "QSR_RestaurantBrands",
    "DPZ": "DPZ_Dominos",
    "BLMN": "BLMN_BloombergME",
    "NWL": "NWL_Newell",
    "RRR": "RRR_RareMedica",
    "DASH": "DASH_DoorDash",
    "LYFT": "LYFT_Lyft",
    "UBER": "UBER_Uber",
    "TGT": "TGT_Target",
    "M": "M_Macys",
    "LOW": "LOW_Lowes",
    "ROST": "ROST_RossStores",
    "BBY": "BBY_BestBuy",
    "NEE": "NEE_NextEra",
    "DUK": "DUK_Duke",
    "SO": "SO_SouthernCo",
    "AEP": "AEP_AmericanElectric",
    "EXC": "EXC_Exelon",
    "SRE": "SRE_Sempra",
    "ES": "ES_Evergy",
    "XEL": "XEL_Xcel",
    "PPL": "PPL_PPL",
    "TMUS": "TMUS_TMobileUS",
    "CHTR": "CHTR_Charter",
    "VOD": "VOD_Vodafone",
    "TM": "TM_Telephone",
    "LOGI": "LOGI_Logitech",
    "NET": "NET_Cloudflare",
    "DDOG": "DDOG_Datadog",
    "SLV": "SLV_Silver",
    "UNG": "UNG_Gas",
    "DBC": "DBC_Commodity",
    "DBA": "DBA_Agri",
    "GDX": "GDX_GoldMiners",
    "GDXJ": "GDXJ_JrMiners",
    "PDBC": "PDBC_Commodity2",
    "CORN": "CORN_Corn",
    "SOYB": "SOYB_Soybean",
    "CBOT_W": "Wheat",
    "IEF": "IEF_MidBond",
    "SHY": "SHY_ShortBond",
    "SHV": "SHV_TBill",
    "BIL": "BIL_TBill3M",
    "AGG": "AGG_Aggregate",
    "BND": "BND_TotalBond",
    "JNK": "JNK_HY2",
    "VCIT": "VCIT_CorpIG",
    "VCSH": "VCSH_CorpST",
    "EMB": "EMB_EM",
    "MBB": "MBB_Mortgage",
    "TIP": "TIP_TIPS",
    "BNDX": "BNDX_IntlBond",
    "HYLD": "HYLD_HYieldETF",
    "PFFA": "PFFA_PreferredA",
    "EWJ": "EWJ_Japan",
    "EWG": "EWG_Germany",
    "EWU": "EWU_UK",
    "EWA": "EWA_Australia",
    "EWH": "EWH_HongKong",
    "EWL": "EWL_Switzerland",
    "EWP": "EWP_Spain",
    "EWI": "EWI_Italy",
    "EWQ": "EWQ_France",
    "EWT": "EWT_Taiwan",
    "EWY": "EWY_Korea",
    "EWZ": "EWZ_Brazil",
    "EWC": "EWC_Canada",
    "EWS": "EWS_Singapore",
    "EWM": "EWM_Malaysia",
    "FXI": "FXI_China",
    "MCHI": "MCHI_China2",
    "IEMG": "IEMG_EM",
    "EEM": "EEM_EM2",
    "VEA": "VEA_DM",
    "INDA": "INDA_India",
    "EPI": "EPI_India2",
    "ASHR": "ASHR_China_A",
    "TUR": "TUR_Turkey",
    "EIDO": "EIDO_Indonesia",
    "EPOL": "EPOL_Poland",
    "EZA": "EZA_SouthAfrica",
    "GXG": "GXG_Germany2",
    "EGRX": "EGRX_Greece",
    "FXB": "FXB_GBP",
    "FXA": "FXA_AUD",
    "FXC": "FXC_CAD",
    "FXF": "FXF_CHF",
    "CEW": "CEW_EM_FX",
    "CYB": "CYB_ChineseYuan",
    "BZF": "BZF_BrazilReal",
    "FXD": "FXD_SwedishKrona",
    "FXN": "FXN_NorwegianKrone",
    "GBTC": "GBTC_Bitcoin",
    "IBIT": "IBIT_Bitcoin2",
    "COIN": "COIN_Crypto",
    "MSTR": "MSTR_Bitcoin3",
    "BITO": "BITO_BitcoinETF",
    "ETHA": "ETHA_EthereumETF",
    "MARA": "MARA_Marathon",
    "RIOT": "RIOT_Riot",
    "CLSK": "CLSK_CleanSpark",
    "CIFR": "CIFR_Cipher",
    "CORZ": "CORZ_Core_Sci",
    "IWM": "IWM_SmallCap",
    "IVV": "IVV_SP500",
    "VTI": "VTI_Total",
    "VOO": "VOO_SP500_2",
    "VV": "VV_LargeCap",
    "VTV": "VTV_Value",
    "VUG": "VUG_Growth",
    "VB": "VB_SmallCap2",
    "SCHD": "SCHD_Div",
    "VIG": "VIG_DivGrowth",
    "HDV": "HDV_HighDiv",
    "NOBL": "NOBL_Aristocrat",
    "DGRO": "DGRO_DividendGrowth",
    "QUAL": "QUAL_Quality",
    "VLUE": "VLUE_Value",
    "VYMI": "VYMI_HighDivYield",
    "JEPI": "JEPI_EquityPremiumIncome",
    "XYLD": "XYLD_XYieldETF",
    "QYLD": "QYLD_NasdaqYield",
    "RYLD": "RYLD_Russell2000Yield",
    "ARKK": "ARKK_Innovation",
    "XBI": "XBI_Biotech",
    "SOXX": "SOXX_Semis",
    "IBB": "IBB_Biotech2",
    "IYT": "IYT_Transport",
    "XHB": "XHB_Homebuilders",
    "KRE": "KRE_RegionalBanks",
    "KBE": "KBE_Banks",
    "ITA": "ITA_Defense",
    "XOP": "XOP_OilExploration",
    "OIH": "OIH_OilServices",
    "IYM": "IYM_BasicMaterials",
    "PCAR": "PCAR_PaccarInc",
    "DAL": "DAL_Delta",
    "AAL": "AAL_AmericanAir",
    "UAL": "UAL_UnitedAir",
    "LUV": "LUV_SouthwestAir",
    "ICLN": "ICLN_CleanEnergy",
    "TAN": "TAN_SolarEnergy",
    "MTUM": "MTUM_Momentum",
    "USMV": "USMV_MinVol",
    "SPLV": "SPLV_LowVol_SP500",
    "RSP": "RSP_EqualWeight_SP500",
    "EUSA": "EUSA_EuropeMomentum",
    "EEMV": "EEMV_EMMinVol",
    "VNQ": "VNQ_US_REIT",
    "IYR": "IYR_US_REIT2",
    "REM": "REM_Mortgage_REIT",
    "SPG": "SPG_SimonProperty",
    "AVB": "AVB_AvalonBay",
    "COLD": "COLD_ColdStorage",
    "DLR": "DLR_Digital_Realty",
    "REXR": "REXR_Rexford",
    "HII": "HII_HuntingtonIngalls",
    "L3HARRIS": "L3H_L3Harris",
    "LDOS": "LDOS_LeadosSecurity",
    "EBAY": "EBAY_eBay",
    "MELI": "MELI_MercadoLibre",
    "SHOP": "SHOP_Shopify",
    "SE": "SE_SeaLimited",
    "PDD": "PDD_PinDuoDuo",
    "JD": "JD_JD.com",
    "VIPS": "VIPS_Vipshop",
    "UPST": "UPST_Upstart",
    "RBLX": "RBLX_Roblox",
    "SNOW": "SNOW_Snowflake",
    "CRWD": "CRWD_CrowdStrike",
    "ZM": "ZM_Zoom",
    "ROKU": "ROKU_Roku",
    "PINS": "PINS_Pinterest",
    "SNAP": "SNAP_Snapchat",
    "TERM": "TERM_Terminal",
    "SPCE": "SPCE_VirginGalactic",
}

YF_TICKERS_RAW = """
    ^GSPC ^IXIC ^DJI ^RUT ^VIX ^VXN ^OVX ^GVZ ^EVZ
    ^FTSE ^N225 ^HSI ^GDAXI ^FCHI ^STOXX50E
    SPY QQQ TLT GLD USO UUP FXE FXY HYG LQD
    AAPL MSFT GOOG AMZN NVDA TSLA JPM JNJ V MA PG UNH HD KO PEP T SMFG DIS XOM CVX BAC WMT VZ CSCO ORCL CRM AMD NFLX ADBE INTC CMCSA PFE ABT LLY DHR COST CMG SBUX MCD ACN PYPL QCOM TXN BA GE BABA

    ^BVSP ^AXJO ^AORD ^IBEX VXX UVXY VIXY SVXY VXZ VIXM XLK XLF XLE XLV XLU XLP XLI XLY XLRE
    XLB XLC GOOGL META AVGO ASML WFC GS BLK SCHW MS COF BAX AXP EQR PLD AMT EQIX CCI PSA ABBV
    MRK BMY AMGN GILD BNTX MRNA CRSP VRTX ILMN DXCM TDOC CI HUM RTX LMT NOC GD CAT DE ITT
    PAYX CTAS MMM HON ETN EMR OTIS JCI PTC SMCI COP SLB EOG MPC PSX VLO PM MO BTI TBP BP TTE
    ENB MET ADM MKC SJM CPB GIS MDLZ NSRGY TAP BDX CLX CL UL LVRK YUM QSR DPZ BLMN NWL RRR
    DASH LYFT UBER TGT M LOW ROST BBY NEE DUK SO AEP EXC SRE ES XEL PPL TMUS CHTR VOD TM LOGI
    NET DDOG SLV UNG DBC DBA GDX GDXJ PDBC CORN SOYB CBOT_W IEF SHY SHV BIL AGG BND JNK VCIT
    VCSH EMB MBB TIP BNDX HYLD PFFA EWJ EWG EWU EWA EWH EWL EWP EWI EWQ EWT EWY EWZ EWC EWS
    EWM FXI MCHI IEMG EEM VEA INDA EPI ASHR TUR EIDO EPOL EZA GXG EGRX FXB FXA FXC FXF CEW
    CYB BZF FXD FXN GBTC IBIT COIN MSTR BITO ETHA MARA RIOT CLSK CIFR CORZ IWM IVV VTI VOO VV
    VTV VUG VB SCHD VIG HDV NOBL DGRO QUAL VLUE VYMI JEPI XYLD QYLD RYLD ARKK XBI SOXX IBB
    IYT XHB KRE KBE ITA XOP OIH IYM PCAR DAL AAL UAL LUV ICLN TAN MTUM USMV SPLV RSP EUSA
    EEMV VNQ IYR REM SPG AVB COLD DLR REXR HII L3HARRIS LDOS EBAY MELI SHOP SE PDD JD VIPS
    UPST RBLX SNOW CRWD ZM ROKU PINS SNAP TERM SPCE
"""

def sanitize_name(ticker: str) -> str:
    name = re.sub(r"[^A-Za-z0-9]+", "_", ticker.replace("^", "IDX_"))
    return name.strip("_")


def build_yf_dict() -> Dict[str, str]:
    tickers = []

    for t in YF_TICKERS_RAW.split():
        t = t.strip()
        if not t:
            continue
        if t in BAD_TICKERS:
            continue
        tickers.append(t)

    seen = set()
    unique_tickers = []

    for t in tickers:
        if t not in seen:
            seen.add(t)
            unique_tickers.append(t)

    yf_dict = {}
    used_names = set()

    for ticker in unique_tickers:
        base_name = MANUAL_YF_NAMES.get(ticker, sanitize_name(ticker))
        name = base_name
        i = 2

        while name in used_names:
            name = f"{base_name}_{i}"
            i += 1

        used_names.add(name)
        yf_dict[ticker] = name

    return yf_dict


# =============================================================================
# FRED INDICATORS
# =============================================================================

fred_dict = {
    "VIXCLS": "VIX",
    "VIXDVOL": "VIX_DrawVol",
    "OILPRICE": "Oil_Price",
    "SP500": "SP500_Level",
    "WILL5000IND": "Wilshire5000",

    "DCOILWTICO": "WTI_Oil_FRED",
    "DCOILBRENTEU": "Brent_Oil_FRED",

    "DGS30": "US30Y_Rate",
    "DGS20": "US20Y_Rate",
    "DGS10": "US10Y_Rate",
    "DGS7": "US7Y_Rate",
    "DGS5": "US5Y_Rate",
    "DGS3": "US3Y_Rate",
    "DGS2": "US2Y_Rate",
    "DGS1": "US1Y_Rate",
    "DTB6": "US6M_Rate",
    "DTB3": "US3M_Rate",
    "DTB1": "US1M_Rate",

    "FEDFUNDS": "FedFunds",
    "EFFR": "EFFR",
    "SOFR": "SOFR_SecuredOIS",
    "DFF": "DFF",

    "T10Y2Y": "T10Y2Y_Spread",
    "T10Y3M": "T10Y3M_Spread",
    "T10YIE": "T10Y_Inflation_Expectation",
    "T5YIE": "T5Y_Inflation_Expectation",
    "T5YIFR": "T5Y5Y_Inflation_Forward",
    "TEDRATE": "TED_Spread",
    "BAMLH0A0HYM2": "HY_OAS",
    "BAMLC0A0CM": "IG_OAS",
    "BAMLC0A4CBBB": "BBB_OAS",

    "UNRATE": "Unemployment",
    "PAYEMS": "NonfarmPayrolls",
    "CPIAUCSL": "CPI",
    "CPILFESL": "Core_CPI",
    "PCE": "PCE",
    "PCEPILFE": "Core_PCE",
    "GDP": "GDP",
    "INDPRO": "Industrial_Production",
    "UMCSENT": "Michigan_Sentiment",
    "RSAFS": "Retail_Sales",

    "NFCI": "NFCI",
    "STLFSI4": "STLFSI4",
}

In [33]:
def safe_series(df: pd.DataFrame, col: str) -> pd.Series:

    x = df.loc[:, col]

    # si plusieurs colonnes (cas MultiIndex / duplicates)
    if isinstance(x, pd.DataFrame):
        x = x.iloc[:, 0]

    # conversion safe
    x = pd.to_numeric(x, errors="coerce")

    # garantit alignement index
    x.index = df.index

    return x


def safe_bool_series(x: pd.Series) -> pd.Series:
    """
    Force une Series en bool propre.
    Évite le bug ~True = -2 / ~False = -1 si dtype=object.
    """
    return x.fillna(False).astype(bool)


In [34]:
# =============================================================================
# DATA LOADER
# =============================================================================

class DataLoader:
    def __init__(self):
        self.yf_failed = []
        self.fred_failed = []

    def load_yfinance_massive(self, tickers: Dict[str, str], start_date: str, end_date: str) -> pd.DataFrame:
        all_tickers = list(tickers.keys())
        chunks = [
            all_tickers[i:i + YF_CHUNK_SIZE]
            for i in range(0, len(all_tickers), YF_CHUNK_SIZE)
        ]

        all_data = []

        for idx, chunk in enumerate(chunks, 1):
            print(f"[DATA] Yahoo chunk {idx}/{len(chunks)} | tickers={len(chunk)}")

            try:
                raw = yf.download(
                    tickers=chunk,
                    start=start_date,
                    end=end_date,
                    progress=False,
                    auto_adjust=False,
                    group_by="ticker",
                    threads=True
                )

                if raw is None or raw.empty:
                    self.yf_failed.extend(chunk)
                    continue

                for ticker in chunk:
                    try:
                        col_name = tickers[ticker]

                        if isinstance(raw.columns, pd.MultiIndex):
                            if ticker not in raw.columns.get_level_values(0):
                                self.yf_failed.append(ticker)
                                continue

                            ticker_df = raw[ticker]

                            if "Close" not in ticker_df.columns:
                                self.yf_failed.append(ticker)
                                continue

                            close = ticker_df["Close"]

                        else:
                            if len(chunk) != 1 or "Close" not in raw.columns:
                                self.yf_failed.append(ticker)
                                continue

                            close = raw["Close"]

                        close = pd.to_numeric(close, errors="coerce")
                        close = close.rename(col_name).to_frame()
                        close.index = pd.to_datetime(close.index)
                        close = close[~close.index.duplicated(keep="last")]
                        close = close.sort_index()

                        if close.dropna().shape[0] >= 100:
                            all_data.append(close)
                        else:
                            self.yf_failed.append(ticker)

                    except Exception:
                        self.yf_failed.append(ticker)

            except Exception:
                self.yf_failed.extend(chunk)

            time.sleep(SLEEP_BETWEEN_CHUNKS)

        if not all_data:
            return pd.DataFrame()

        df = pd.concat(all_data, axis=1, join="outer").sort_index()
        df = df.loc[:, ~df.columns.duplicated()]
        return df

    def load_fred(self, indicators: Dict[str, str], start_date: str, end_date: str) -> pd.DataFrame:
        data = []

        for i, (code, col_name) in enumerate(indicators.items(), 1):
            print(f"[DATA] FRED {i}/{len(indicators)} | {code}")

            try:
                raw = pdr.get_data_fred(code, start=start_date, end=end_date)

                if raw is None or raw.empty:
                    self.fred_failed.append(code)
                    continue

                series = raw.iloc[:, 0]
                series = pd.to_numeric(series, errors="coerce")
                series = series.rename(col_name).to_frame()
                series.index = pd.to_datetime(series.index)
                series = series[~series.index.duplicated(keep="last")]
                series = series.sort_index()

                if series.dropna().shape[0] >= 30:
                    data.append(series)
                else:
                    self.fred_failed.append(code)

            except Exception:
                self.fred_failed.append(code)

            time.sleep(0.1)

        if not data:
            return pd.DataFrame()

        df = pd.concat(data, axis=1, join="outer").sort_index()
        df = df.loc[:, ~df.columns.duplicated()]
        return df

    def combine_to_latest_full_dataset(self, yf_df: pd.DataFrame, fred_df: pd.DataFrame) -> pd.DataFrame:
        """
        CORRECTIF (bug identifié) : l'ancienne version calculait la couverture
        de chaque colonne (notna().mean()) puis filtrait les LIGNES à >=95% de
        couverture. Comme plusieurs colonnes (tickers/indices créés après 2000,
        ex: OVX 2007, GVZ 2008) n'ont pas d'historique avant ~2010, la
        couverture moyenne des lignes < 2010 tombait sous le seuil et TOUTES
        les lignes pré-2010 étaient supprimées - peu importe MIN_DATA_START.
        Conséquence concrète : tout le sweep TRAIN_START=2000..2010 utilisait
        en réalité toujours la même fenêtre (~2010+), donc les 22 runs du
        sweep étaient quasi identiques entre eux.

        CORRECTIF appliqué : on calcule la couverture PAR COLONNE sur toute la
        fenêtre demandée (depuis MIN_DATA_START), et on retire en amont les
        colonnes dont la couverture est insuffisante - PAS les lignes. Une
        colonne sans historique avant 2010 est donc exclue du feature set
        pour ce run, mais les lignes 2000-2009 sont conservées avec les
        features qui, elles, couvrent bien toute la période.
        """
        if yf_df.empty:
            raise ValueError("Yahoo Finance data is empty.")

        yf_df = yf_df.sort_index()
        yf_df.index = pd.to_datetime(yf_df.index)
        yf_df = yf_df.loc[yf_df.index >= MIN_DATA_START].copy()
        yf_df = yf_df.dropna(axis=1, how="all")

        if yf_df.empty:
            raise ValueError("Yahoo Finance dataframe has no usable columns since MIN_DATA_START.")

        if not fred_df.empty:
            fred_df = fred_df.sort_index()
            fred_df.index = pd.to_datetime(fred_df.index)
            fred_df = fred_df.loc[fred_df.index >= MIN_DATA_START].copy()
            fred_df = fred_df.dropna(axis=1, how="all")

            fred_on_market_calendar = fred_df.reindex(yf_df.index).ffill()
            combined = pd.concat([yf_df, fred_on_market_calendar], axis=1)
        else:
            combined = yf_df.copy()

        combined = combined.loc[:, ~combined.columns.duplicated()]
        combined = combined.replace([np.inf, -np.inf], np.nan)
        combined = combined.sort_index()

        if "VIX_Price" not in combined.columns and "VIX" not in combined.columns:
            raise ValueError("No usable VIX column found after combining data.")

        # --- Filtre 1 : colonne apparue trop tard après MIN_DATA_START ---
        latest_allowed_first_valid = MIN_DATA_START + pd.Timedelta(days=MAX_FIRST_VALID_LAG_DAYS)

        keep_cols_start = []
        for col in combined.columns:
            first_valid = combined[col].first_valid_index()
            if first_valid is None:
                continue
            if first_valid <= latest_allowed_first_valid:
                keep_cols_start.append(col)

        combined = combined[keep_cols_start]

        if combined.empty:
            raise ValueError("No columns left after first-valid-date filter.")

        combined = combined.ffill()

        # --- Filtre 2 (CORRIGÉ) : couverture par COLONNE sur toute la fenêtre,
        # pas de filtre de ligne. Seuil MIN_COLUMN_COVERAGE (ex: 0.90). ---
        coverage = combined.notna().mean()
        keep_cols_coverage = coverage[coverage >= MIN_COLUMN_COVERAGE].index.tolist()

        core_cols = [
            "VIX_Price",
            "VIX",
            "SP500_Price",
            "SPY",
            "QQQ",
            "TLT_LongBond",
            "GLD_Gold",
            "USO_Oil"
        ]

        for c in core_cols:
            if c in combined.columns and c not in keep_cols_coverage:
                if combined[c].notna().mean() >= 0.75:
                    keep_cols_coverage.append(c)

        n_cols_before = combined.shape[1]
        combined = combined[keep_cols_coverage]
        print(f"[COMBINE] Colonnes retirées par couverture insuffisante (<{MIN_COLUMN_COVERAGE:.0%}): "
              f"{n_cols_before - combined.shape[1]}/{n_cols_before}")

        if combined.empty:
            raise ValueError("No columns left after coverage filter.")

        # --- PAS de filtre de ligne (row_coverage) ici : on garde toutes les
        # lignes depuis MIN_DATA_START, puisque les colonnes retenues couvrent
        # déjà >= MIN_COLUMN_COVERAGE de cette fenêtre par construction. ---
        combined = combined.ffill()
        combined = combined.dropna(axis=0, how="any")

        if combined.empty:
            raise ValueError("Final dataset is empty after coverage filters.")

        print(f"[COMBINE] Columns kept: {combined.shape[1]}")
        print(f"[COMBINE] Rows kept:    {combined.shape[0]}")
        print(f"[COMBINE] Date range:   {combined.index.min().date()} → {combined.index.max().date()}")

        return combined


In [35]:
class FeatureEngineer:
    def __init__(self):
        self.features = []

    def add(self, name: str):
        if name not in self.features:
            self.features.append(name)

    def create_features(self, df: pd.DataFrame):
        print("[FEATURES] Creating features...")

        df = df.copy()
        df = df.loc[:, ~df.columns.duplicated()]
        base_cols = list(df.columns)

        for col in base_cols:
            s = safe_series(df, col)
            # Replace 0 values with NaN to avoid ZeroDivisionError in pct_change
            s_clean = s.replace(0, np.nan)

            # returns
            for p in [1, 5, 20]:
                f = f"{col}_ret_{p}d"
                df[f] = s_clean.pct_change(p)
                self.add(f)

            # volatility
            f = f"{col}_vol_20d"
            df[f] = s_clean.pct_change().rolling(20).std()
            self.add(f)

            # z-score
            mean = s_clean.rolling(60).mean()
            std = s_clean.rolling(60).std()
            f = f"{col}_zscore_60d"
            df[f] = (s_clean - mean) / (std + 1e-8)
            self.add(f)

        # VIX features
        vix_col = "VIX_Price" if "VIX_Price" in df.columns else "VIX" if "VIX" in df.columns else None

        if vix_col:
            vix = safe_series(df, vix_col)

            df["vix_level"] = vix
            df["vix_change_1d"] = vix.pct_change(1)
            df["vix_change_5d"] = vix.pct_change(5)
            df["vix_ma_20"] = vix.rolling(20).mean()
            df["vix_vs_ma20"] = vix - df["vix_ma_20"]

            for f in [
                "vix_level",
                "vix_change_1d",
                "vix_change_5d",
                "vix_ma_20",
                "vix_vs_ma20"
            ]:
                self.add(f)

        # SP500 features
        if "SP500_Price" in df.columns:
            spx = safe_series(df, "SP500_Price")

            df["spx_realized_vol_20d"] = spx.pct_change().rolling(20).std() * np.sqrt(252)

            # Fix for spx_drawdown_252d to prevent data leakage:
            # Calculate the peak from the *previous* 252 days, excluding the current day.
            # This ensures the feature for day 't' only uses data available up to day 't-1'.
            spx_peak_before_today = spx.rolling(window=252, closed='left').max()
            df["spx_drawdown_252d"] = (spx_peak_before_today - spx) / (spx_peak_before_today + 1e-8)
            df["spx_drawdown_252d"] = df["spx_drawdown_252d"].clip(lower=0) # Drawdown cannot be negative

            df["spx_down_day"] = (spx.pct_change(1) < 0).astype(int)

            for f in [
                "spx_realized_vol_20d",
                "spx_drawdown_252d",
                "spx_down_day"
            ]:
                self.add(f)

        # Yield curve
        if "US10Y_Rate" in df.columns and "US2Y_Rate" in df.columns:
            us10 = safe_series(df, "US10Y_Rate")
            us2 = safe_series(df, "US2Y_Rate")

            df["yield_curve_10y_2y"] = us10 - us2
            df["yield_curve_change_20d"] = df["yield_curve_10y_2y"].diff(20)

            for f in [
                "yield_curve_10y_2y",
                "yield_curve_change_20d"
            ]:
                self.add(f)

        self.features = [f for f in self.features if f in df.columns]

        return df, self.features

In [36]:
class TargetBuilder:
    """
    Construit la cible (VIX_Direction) et le régime de marché (VIX_Regime).

    horizon_days controle l'horizon de prediction : la cible devient
    "le VIX monte-t-il entre T et T+horizon_days" au lieu de T et T+1 fixe.
    Avec horizon_days=1 (defaut), comportement strictement identique a avant.

    Deux modes pour définir les seuils de régime CALM/NORMAL/STRESS :

    - mode="fixed"   : quantiles q33/q67 calculés UNE FOIS sur la période
                        train (< train_end), puis appliqués tels quels à
                        tout le dataframe (train + test). Pas de fuite
                        train->test, mais le seuil ne s'adapte pas si le
                        régime de volatilité change structurellement avec
                        le temps (ex: VIX 2008 vs VIX 2017).

    - mode="rolling" : quantiles q33/q67 recalculés à CHAQUE date T sur une
                        fenêtre glissante des `rolling_window` jours
                        précédents (closed='left', donc strictement avant T
                        - pas de fuite intra-jour). Le régime à la date T
                        reflète le niveau de VIX relatif à son contexte
                        récent (~2 ans avec rolling_window=504), pas à toute
                        l'histoire 2000-2026 mélangée.
    """
    def __init__(self, q_low: float = 0.33, q_high: float = 0.67,
                 mode: str = "fixed", rolling_window: int = 504,
                 horizon_days: int = 1):
        assert mode in ("fixed", "rolling"), "mode must be 'fixed' or 'rolling'"
        assert horizon_days >= 1, "horizon_days must be >= 1"
        self.vix_col = None
        self.q_low = q_low
        self.q_high = q_high
        self.mode = mode
        self.rolling_window = rolling_window
        self.horizon_days = horizon_days  # horizon de prediction en jours de bourse (1, 3, 5...)
        self.calm_threshold_ = None    # scalar if mode="fixed", else None
        self.stress_threshold_ = None
        self.flat_threshold = VIX_FLAT_PCT_THRESHOLD
        self.flat_removed = 0
        self.total_before_flat_filter = 0

    def build(self, df: pd.DataFrame, train_end: str = None) -> pd.DataFrame:
        df = df.copy()
        df = df.loc[:, ~df.columns.duplicated()]

        if "VIX_Price" in df.columns:
            self.vix_col = "VIX_Price"
        elif "VIX" in df.columns:
            self.vix_col = "VIX"
        else:
            raise KeyError("No VIX column found.")

        vix = safe_series(df, self.vix_col)
        # Horizon de prediction : shift(-horizon_days) au lieu de shift(-1) fixe.
        # horizon_days=1 reproduit exactement le comportement d'origine.
        future_vix = vix.shift(-self.horizon_days)

        vix_next_change = (future_vix / vix) - 1

        df["VIX_Next_Change"] = vix_next_change
        df["VIX_Is_Flat"] = vix_next_change.abs() <= self.flat_threshold

        direction = pd.Series(np.nan, index=df.index)
        direction.loc[vix_next_change > 0] = 1
        direction.loc[vix_next_change <= 0] = 0
        direction.loc[future_vix.isna()] = np.nan

        df["VIX_Direction"] = direction
        df.loc[future_vix.isna(), "VIX_Is_Flat"] = np.nan

        if self.mode == "fixed":
            # --- Quantiles fixes, calculés sur le train seulement (no leakage) ---
            if train_end is not None:
                vix_for_quantiles = vix.loc[vix.index < pd.Timestamp(train_end)]
            else:
                vix_for_quantiles = vix

            self.calm_threshold_ = vix_for_quantiles.quantile(self.q_low)
            self.stress_threshold_ = vix_for_quantiles.quantile(self.q_high)

            regime = pd.Series("NORMAL", index=df.index)
            regime.loc[vix < self.calm_threshold_] = "CALM"
            regime.loc[vix >= self.stress_threshold_] = "STRESS"

            threshold_desc = (
                f"CALM < {self.calm_threshold_:.2f}, "
                f"NORMAL [{self.calm_threshold_:.2f}-{self.stress_threshold_:.2f}), "
                f"STRESS >= {self.stress_threshold_:.2f}  (fixe, calculé sur train)"
            )

        else:
            # --- Quantiles rolling : recalculés à chaque date T sur les
            # `rolling_window` jours STRICTEMENT précédents (closed='left').
            # Pas de fuite : le quantile à T n'utilise jamais VIX(T) ni le futur.
            rolling_calm = vix.rolling(window=self.rolling_window, closed='left').quantile(self.q_low)
            rolling_stress = vix.rolling(window=self.rolling_window, closed='left').quantile(self.q_high)

            self.calm_threshold_ = rolling_calm   # Series, pas un scalaire
            self.stress_threshold_ = rolling_stress

            regime = pd.Series("NORMAL", index=df.index)
            regime.loc[vix < rolling_calm] = "CALM"
            regime.loc[vix >= rolling_stress] = "STRESS"
            # Tant que la fenêtre rolling n'est pas pleine (début d'historique),
            # rolling_calm/rolling_stress sont NaN -> régime indéfini -> ces
            # lignes seront retirées plus bas (dropna sur VIX_Regime_valid).
            regime.loc[rolling_calm.isna() | rolling_stress.isna()] = np.nan

            threshold_desc = (
                f"rolling sur {self.rolling_window} jours (~{self.rolling_window/252:.1f} ans), "
                f"recalculé à chaque date T sur les jours STRICTEMENT antérieurs à T"
            )

        df["VIX_Regime"] = regime

        # Lignes à retirer : direction NaN (fin de série), OU régime NaN (mode
        # rolling, début de série sans assez d'historique pour la fenêtre)
        df = df.dropna(subset=["VIX_Direction", "VIX_Is_Flat", "VIX_Regime"]).copy()

        df["VIX_Is_Flat"] = df["VIX_Is_Flat"].fillna(False).astype(bool)

        self.total_before_flat_filter = int(len(df))
        self.flat_removed = int(df["VIX_Is_Flat"].sum())

        df = df.loc[~df["VIX_Is_Flat"]].copy()

        df["VIX_Direction"] = df["VIX_Direction"].astype(int)
        df["VIX_Is_Flat"] = df["VIX_Is_Flat"].astype(bool)

        print(f"[TARGET] VIX column: {self.vix_col}  |  mode={self.mode}")
        print(f"[TARGET] VIX regime thresholds: {threshold_desc}")
        print(f"[TARGET] VIX flat threshold: ±{self.flat_threshold:.2%}")
        print(f"[TARGET] Flat days removed completely: {self.flat_removed}/{self.total_before_flat_filter}")
        print(f"[TARGET] Remaining non-flat rows: {len(df)}")
        print(df["VIX_Regime"].value_counts())

        return df



In [37]:
class TrainFittedCleaner:
    def __init__(self):
        self.medians = None
        self.lower = None
        self.upper = None

    def fit(self, X: pd.DataFrame):
        X = X.replace([np.inf, -np.inf], np.nan)
        self.medians = X.median()
        self.lower = X.quantile(0.01)
        self.upper = X.quantile(0.99)
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        X = X.copy()
        X = X.replace([np.inf, -np.inf], np.nan)
        X = X.fillna(self.medians)
        X = X.clip(lower=self.lower, upper=self.upper, axis=1)
        X = X.fillna(0)
        return X

    def fit_transform(self, X: pd.DataFrame) -> pd.DataFrame:
        self.fit(X)
        return self.transform(X)

In [38]:
def evaluate_model_across_regimes(df_test, trained_by_regime):
    all_preds = pd.Series(index=df_test.index, dtype=float)
    all_probas = pd.Series(index=df_test.index, dtype=float)

    for regime, pack in trained_by_regime.items():
        mask = df_test["VIX_Regime"] == regime

        if mask.sum() == 0:
            continue

        X_raw = df_test.loc[mask, pack["features"]]
        X_clean = pack["cleaner"].transform(X_raw)
        X_scaled = pack["scaler"].transform(X_clean)

        model = pack["model"]

        pred = model.predict(X_scaled)

        if hasattr(model, "predict_proba"):
            proba = model.predict_proba(X_scaled)[:, 1]
        else:
            proba = pred.astype(float)

        all_preds.loc[mask] = pred
        all_probas.loc[mask] = proba

    valid = all_preds.notna()

    y_true = df_test.loc[valid, "VIX_Direction"].values
    y_pred = all_preds.loc[valid].astype(int).values
    y_proba = all_probas.loc[valid].values

    if len(y_true) == 0:
        return None

    return compute_metrics(y_true, y_pred, y_proba)

In [39]:
def model_configs():
    return {
        "XGBoost": (
            XGBClassifier,
            {
                "max_depth": [2, 3],
                "learning_rate": [0.03, 0.05],
                "n_estimators": [75, 125],
                "subsample": [0.8],
                "colsample_bytree": [0.8]
            },
            {
                "random_state": RANDOM_STATE,
                "eval_metric": "logloss",
                "n_jobs": -1
            }
        ),
        "LightGBM": (
            LGBMClassifier,
            {
                "num_leaves": [7, 15],
                "learning_rate": [0.03, 0.05],
                "n_estimators": [75, 125],
                "max_depth": [3, 5]
            },
            {
                "random_state": RANDOM_STATE,
                "verbose": -1,
                "class_weight": "balanced"
            }
        ),
        "GradientBoosting": (
            GradientBoostingClassifier,
            {
                "n_estimators": [75, 125],
                "learning_rate": [0.03, 0.05],
                "max_depth": [2, 3],
                "min_samples_leaf": [10]
            },
            {
                "random_state": RANDOM_STATE
            }
        ),
        "RandomForest": (
            RandomForestClassifier,
            {
                "n_estimators": [150],
                "max_depth": [3, 5],
                "min_samples_leaf": [10, 20]
            },
            {
                "random_state": RANDOM_STATE,
                "n_jobs": -1,
                "class_weight": "balanced"
            }
        ),
        "LogisticRegression": (
            LogisticRegression,
            {
                "C": [0.01, 0.1, 1.0],
                "penalty": ["l2"]
            },
            {
                "random_state": RANDOM_STATE,
                "max_iter": 2000,
                "class_weight": "balanced"
            }
        ),
    }


In [40]:
def compute_metrics(y_true, y_pred, y_proba):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "R2": r2_score(y_true, y_proba),
        "AUC": roc_auc_score(y_true, y_proba) if len(np.unique(y_true)) > 1 else np.nan,
        "TP": int(tp),
        "FP": int(fp),
        "TN": int(tn),
        "FN": int(fn),
        "Pred_0": int((y_pred == 0).sum()),
        "Pred_1": int((y_pred == 1).sum()),
        "Actual_0": int((y_true == 0).sum()),
        "Actual_1": int((y_true == 1).sum()),
        "Confusion_Matrix": cm.tolist()
    }

In [41]:
print("[INIT] Building ticker dictionaries...")
yf_dict = build_yf_dict()
print(f"[INIT] Yahoo tickers: {len(yf_dict)}")
print(f"[INIT] FRED indicators: {len(fred_dict)}")

[INIT] Building ticker dictionaries...
[INIT] Yahoo tickers: 345
[INIT] FRED indicators: 43


In [42]:
loader = DataLoader()
end_date = pd.Timestamp.today().strftime("%Y-%m-%d")
print("[STEP 1/6] Loading data...")
yf_df = loader.load_yfinance_massive(yf_dict, TRAIN_START, end_date)
fred_df = loader.load_fred(fred_dict, TRAIN_START, end_date)

print("[STEP 2/6] Combining data...")
df = loader.combine_to_latest_full_dataset(yf_df, fred_df)


[STEP 1/6] Loading data...
[DATA] Yahoo chunk 1/9 | tickers=40
[DATA] Yahoo chunk 2/9 | tickers=40
[DATA] Yahoo chunk 3/9 | tickers=40
[DATA] Yahoo chunk 4/9 | tickers=40
[DATA] Yahoo chunk 5/9 | tickers=40


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['LVRK']: YFTzMissingError('possibly delisted; no timezone found')


[DATA] Yahoo chunk 6/9 | tickers=40


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['CBOT_W']: YFTzMissingError('possibly delisted; no timezone found')


[DATA] Yahoo chunk 7/9 | tickers=40


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['BZF']: YFPricesMissingError('possibly delisted; no price data found  (1d 2000-01-01 -> 2026-07-01)')


[DATA] Yahoo chunk 8/9 | tickers=40
[DATA] Yahoo chunk 9/9 | tickers=25


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['L3HARRIS']: YFTzMissingError('possibly delisted; no timezone found')


[DATA] FRED 1/43 | VIXCLS
[DATA] FRED 2/43 | VIXDVOL
[DATA] FRED 3/43 | OILPRICE
[DATA] FRED 4/43 | SP500
[DATA] FRED 5/43 | WILL5000IND
[DATA] FRED 6/43 | DCOILWTICO
[DATA] FRED 7/43 | DCOILBRENTEU
[DATA] FRED 8/43 | DGS30
[DATA] FRED 9/43 | DGS20
[DATA] FRED 10/43 | DGS10
[DATA] FRED 11/43 | DGS7
[DATA] FRED 12/43 | DGS5
[DATA] FRED 13/43 | DGS3
[DATA] FRED 14/43 | DGS2
[DATA] FRED 15/43 | DGS1
[DATA] FRED 16/43 | DTB6
[DATA] FRED 17/43 | DTB3
[DATA] FRED 18/43 | DTB1
[DATA] FRED 19/43 | FEDFUNDS
[DATA] FRED 20/43 | EFFR
[DATA] FRED 21/43 | SOFR
[DATA] FRED 22/43 | DFF
[DATA] FRED 23/43 | T10Y2Y
[DATA] FRED 24/43 | T10Y3M
[DATA] FRED 25/43 | T10YIE
[DATA] FRED 26/43 | T5YIE
[DATA] FRED 27/43 | T5YIFR
[DATA] FRED 28/43 | TEDRATE
[DATA] FRED 29/43 | BAMLH0A0HYM2
[DATA] FRED 30/43 | BAMLC0A0CM
[DATA] FRED 31/43 | BAMLC0A4CBBB
[DATA] FRED 32/43 | UNRATE
[DATA] FRED 33/43 | PAYEMS
[DATA] FRED 34/43 | CPIAUCSL
[DATA] FRED 35/43 | CPILFESL
[DATA] FRED 36/43 | PCE
[DATA] FRED 37/43 | PCEPILF

In [43]:
print("[STEP 3/6] Feature engineering...")
engineer = FeatureEngineer()
df, features = engineer.create_features(df)

[STEP 3/6] Feature engineering...
[FEATURES] Creating features...


In [44]:
# Copie de référence du dataframe après feature engineering, AVANT target/régime.
# Sert de point de départ identique pour les deux modes de quantile (fixed/rolling).
df_post_features = df.copy()
features_post_engineering = list(features)
print(f"[CHECKPOINT] df_post_features: {df_post_features.shape}, features: {len(features_post_engineering)}")


[CHECKPOINT] df_post_features: (6734, 1180), features: 985


In [45]:
print("[STEP] Chargement et feature engineering terminés. "
      "Démarrage de la Phase 1 (scoring des fenêtres par régime).")


[STEP] Chargement et feature engineering terminés. Démarrage de la Phase 1 (scoring des fenêtres par régime).


In [46]:
def model_configs():
    return {
        "XGBoost": (XGBClassifier,
            {"max_depth":[2,3],"learning_rate":[0.03,0.05],
             "n_estimators":[75,125],"subsample":[0.8],"colsample_bytree":[0.8]},
            {"random_state":RANDOM_STATE,"eval_metric":"logloss","n_jobs":-1}),
        "LightGBM": (LGBMClassifier,
            {"num_leaves":[7,15],"learning_rate":[0.03,0.05],
             "n_estimators":[75,125],"max_depth":[3,5]},
            {"random_state":RANDOM_STATE,"verbose":-1,"class_weight":"balanced"}),
        "GradientBoosting": (GradientBoostingClassifier,
            {"n_estimators":[75,125],"learning_rate":[0.03,0.05],
             "max_depth":[2,3],"min_samples_leaf":[10]},
            {"random_state":RANDOM_STATE}),
        "RandomForest": (RandomForestClassifier,
            {"n_estimators":[150],"max_depth":[3,5],"min_samples_leaf":[10,20]},
            {"random_state":RANDOM_STATE,"n_jobs":-1,"class_weight":"balanced"}),
        "LogisticRegression": (LogisticRegression,
            {"C":[0.01,0.1,1.0],"penalty":["l2"]},
            {"random_state":RANDOM_STATE,"max_iter":2000,"class_weight":"balanced"}),
    }

def compute_metrics(y_true, y_pred, y_proba):
    try:   auc = roc_auc_score(y_true, y_proba)
    except: auc = np.nan
    cm = confusion_matrix(y_true, y_pred, labels=[0,1])
    tn,fp,fn,tp = cm.ravel() if cm.size==4 else (0,0,0,0)
    return {
        "Accuracy":  accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall":    recall_score(y_true, y_pred, zero_division=0),
        "F1":        f1_score(y_true, y_pred, zero_division=0),
        "AUC": auc,
        "TP":int(tp),"FP":int(fp),"TN":int(tn),"FN":int(fn),
        "Actual_0":int(tn+fp),"Actual_1":int(tp+fn),
    }


In [47]:
# =============================================================================
# GÉNÉRATION D'INTERACTIONS — deux niveaux
# =============================================================================
# NIVEAU 1 (simple) : pour toutes les paires (i,j) du top-N features :
#   - Ratio   : feat_i / feat_j  (protégé ε=1e-8)
#   - Diff    : feat_i - feat_j
#
# NIVEAU 2 (complexe) : pour les mêmes paires :
#   - Produit        : feat_i × feat_j
#   - Z-score relatif: (feat_i - feat_j) / rolling_std(feat_i - feat_j, w=20)
#   - MA croisée     : rolling_mean(feat_i, w=20) / rolling_mean(feat_j, w=20)
#   - Momentum croisé: feat_i_ret5d × feat_j_level
#     (approximé : feat_i.pct_change(5) × feat_j, uniquement si feat_j est
#      un niveau/zscore, sinon skipped)
# =============================================================================

def generate_all_interactions(df: pd.DataFrame,
                               base_features: list,
                               top_n: int = 20,
                               rolling_w: int = 20,
                               eps: float = 1e-8) -> pd.DataFrame:
    """Génère interactions simples + complexes sur le top_n features.
    Retourne un DataFrame de nouvelles colonnes uniquement (même index que df).
    """
    feats = [f for f in base_features[:top_n] if f in df.columns]
    n = len(feats)
    n_pairs = n*(n-1)//2
    print(f"[INTERACTIONS] {n} features de base → {n_pairs} paires "
          f"× 5 opérations = jusqu'à {n_pairs*5} colonnes")

    new_cols = {}
    for i in range(n):
        for j in range(i+1, n):
            fi, fj = feats[i], feats[j]
            si, sj = df[fi], df[fj]

            # -- Niveau 1 : simples --
            # Ratio
            safe_denom = sj.where(sj.abs() >= eps, np.nan)
            new_cols[f"{fi}__div__{fj}"]   = si / safe_denom

            # Différence
            new_cols[f"{fi}__minus__{fj}"] = si - sj

            # -- Niveau 2 : complexes --
            # Produit
            new_cols[f"{fi}__prod__{fj}"]  = si * sj

            # Z-score relatif : (A-B) / rolling_std(A-B)
            diff_series = si - sj
            roll_std = diff_series.rolling(rolling_w, min_periods=rolling_w//2).std()
            new_cols[f"{fi}__zrel__{fj}"]  = diff_series / roll_std.replace(0, np.nan)

            # MA croisée : rolling_mean(A) / rolling_mean(B)
            ma_i = si.rolling(rolling_w, min_periods=rolling_w//2).mean()
            ma_j = sj.rolling(rolling_w, min_periods=rolling_w//2).mean()
            safe_ma_j = ma_j.where(ma_j.abs() >= eps, np.nan)
            new_cols[f"{fi}__macross__{fj}"] = ma_i / safe_ma_j

            # Momentum croisé : ret_5d(A) × B
            ret5_i = si.pct_change(5)
            new_cols[f"{fi}__ret5x__{fj}"] = ret5_i * sj

    interactions_df = pd.DataFrame(new_cols, index=df.index)
    interactions_df = interactions_df.replace([np.inf, -np.inf], np.nan)

    # Exclure colonnes entièrement NaN
    interactions_df = interactions_df.dropna(axis=1, how="all")
    print(f"[INTERACTIONS] {interactions_df.shape[1]} colonnes générées "
          f"(après suppression des colonnes entièrement NaN)")
    return interactions_df


In [48]:
# =============================================================================
# SÉLECTION SHAP (Phase 1)
# =============================================================================
# Démarche :
#   Étape A : XGBoost pilote sur les features de base → SHAP values → top-40
#   Étape B : génération interactions (simples + complexes) sur le top-20
#   Étape C : XGBoost pilote sur (top-40 base + interactions) → SHAP values
#             → top-30 final (ce sont les features utilisées en Phase 2)
#   Étape D : TRAIN_START = première date où toutes les features finales
#             sont non-NaN (déterminé automatiquement)
#
# La sélection est indépendante par régime ET par horizon.
# =============================================================================

def shap_select_features(X_train: pd.DataFrame,
                          y_train: np.ndarray,
                          top_n: int,
                          label: str = "") -> list:
    """Entraîne un XGBoost pilote léger, calcule les SHAP values (TreeExplainer,
    rapide pour XGBoost), et retourne les top_n features par |mean SHAP|."""
    pilot = XGBClassifier(
        n_estimators=SHAP_PILOT_N_ESTIMATORS,
        max_depth=3, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=RANDOM_STATE, n_jobs=-1
    )
    pilot.fit(X_train.values, y_train)

    explainer = shap.TreeExplainer(pilot)
    shap_values = explainer.shap_values(X_train.values)

    # shap_values peut être un array 2D (classification binaire avec XGBoost)
    if isinstance(shap_values, list):
        shap_arr = np.abs(shap_values[1])   # classe positive
    else:
        shap_arr = np.abs(shap_values)

    mean_abs_shap = pd.Series(shap_arr.mean(axis=0), index=X_train.columns)
    ranked = mean_abs_shap.sort_values(ascending=False)

    top_features = ranked.head(top_n).index.tolist()
    if label:
        print(f"[SHAP {label}] Top-{top_n} sélectionnées sur {len(X_train.columns)} candidates")
    return top_features


def run_phase1_shap(horizon_days: int,
                    df_post_features: pd.DataFrame,
                    features_post_engineering: list):
    """Phase 1 SHAP complète pour un horizon donné.
    Retourne un dict {regime: {"features": [...], "train_start": "YYYY-MM-DD"}}
    """
    print(f"\n{'#'*80}\n# PHASE 1 SHAP — HORIZON = {horizon_days}j\n{'#'*80}")

    result = {}

    for regime in ALL_REGIMES:
        print(f"\n--- PHASE 1 | h={horizon_days}j | {regime} ---")

        # Construire la cible sur tout l'historique disponible,
        # train_end=TEST_DATE pour que les seuils de régime ne voient pas le test
        tb = TargetBuilder(q_low=0.33, q_high=0.67, mode="fixed",
                           rolling_window=ROLLING_QUANTILE_WINDOW,
                           horizon_days=horizon_days)
        df = tb.build(df_post_features, train_end=TEST_DATE)

        # Portion TRAIN uniquement (< 2022) pour la sélection SHAP
        df_train = df.loc[df.index < pd.Timestamp(TEST_DATE)].copy()
        if regime == "GLOBAL":
            df_train_regime = df_train
        else:
            df_train_regime = df_train.loc[df_train["VIX_Regime"] == regime].copy()

        feats_available = [f for f in features_post_engineering
                           if f in df_train_regime.columns]

        if len(df_train_regime) < 50 or len(np.unique(df_train_regime["VIX_Direction"])) < 2:
            print(f"[WARN] {regime}: échantillon trop petit. Skip.")
            continue

        # Nettoyage + standardisation pour le pilote SHAP
        cleaner = TrainFittedCleaner()
        X_base = pd.DataFrame(
            cleaner.fit_transform(df_train_regime[feats_available]),
            columns=feats_available, index=df_train_regime.index
        ).dropna(axis=1, how="all")

        scaler_base = StandardScaler()
        X_base_scaled = pd.DataFrame(
            scaler_base.fit_transform(X_base),
            columns=X_base.columns, index=X_base.index
        )
        y_train = df_train_regime["VIX_Direction"].loc[X_base_scaled.index].values

        # ─── Étape A : SHAP sur features de base → top-40 ─────────────────
        top_base = shap_select_features(
            X_base_scaled, y_train,
            top_n=SHAP_TOP_BASE_N, label=f"{regime} base"
        )

        # ─── Étape B : génération interactions sur top-20 de la base ───────
        df_base_for_interactions = df_train_regime[top_base].dropna(how="all")
        interactions_df = generate_all_interactions(
            df_base_for_interactions,
            base_features=top_base,
            top_n=N_TOP_FOR_INTERACTIONS,
            rolling_w=INTERACTION_ROLLING_WINDOW
        )

        # ─── Étape C : SHAP sur (top-40 base + interactions) → top-30 final
        df_extended = pd.concat([
            df_train_regime[top_base],
            interactions_df
        ], axis=1).loc[:, ~pd.concat([
            df_train_regime[top_base], interactions_df
        ], axis=1).columns.duplicated()]

        extended_features = top_base + list(interactions_df.columns)

        cleaner2 = TrainFittedCleaner()
        X_ext = pd.DataFrame(
            cleaner2.fit_transform(df_extended[extended_features]),
            columns=extended_features, index=df_extended.index
        ).dropna(axis=1, how="all")

        scaler_ext = StandardScaler()
        X_ext_scaled = pd.DataFrame(
            scaler_ext.fit_transform(X_ext),
            columns=X_ext.columns, index=X_ext.index
        )
        y_ext = df_train_regime["VIX_Direction"].loc[X_ext_scaled.index].values

        top_final = shap_select_features(
            X_ext_scaled, y_ext,
            top_n=SHAP_TOP_FINAL_N, label=f"{regime} base+interactions"
        )

        n_interactions_final = sum(
            1 for f in top_final
            if any(sep in f for sep in ["__div__","__minus__","__prod__","__zrel__","__macross__","__ret5x__"])
        )
        print(f"[SHAP FINAL] {regime} h={horizon_days}j: {len(top_final)} features retenues "
              f"({n_interactions_final} interactions, {len(top_final)-n_interactions_final} base)")

        # ─── Étape D : TRAIN_START automatique ─────────────────────────────
        # On reconstruit les features finales sur tout l'historique disponible
        # pour trouver la première date sans NaN.
        df_all = tb.build(df_post_features, train_end=TEST_DATE)
        df_all_ext = pd.concat([
            df_all[top_final if all(f in df_all.columns for f in top_final) else
                   [f for f in top_final if f in df_all.columns]],
        ], axis=1)

        # Features de base présentes directement dans df_all
        base_in_df = [f for f in top_final if f in df_all.columns]
        # Features d'interaction : reconstruire sur df_all
        inter_cols = [f for f in top_final if f not in df_all.columns]

        if inter_cols:
            # Extraire les features SOURCES de chaque nom d'interaction.
            # Format des noms : "feat_A__optype__feat_B"
            # On split sur le premier et le dernier séparateur connu.
            _SEPS = ["__div__","__minus__","__prod__","__zrel__","__macross__","__ret5x__"]
            source_set = set(base_in_df)
            for fname in inter_cols:
                for sep in _SEPS:
                    if sep in fname:
                        parts = fname.split(sep, 1)  # split sur la PREMIÈRE occurrence
                        feat_a = parts[0]
                        feat_b = parts[1]
                        if feat_a in df_all.columns:
                            source_set.add(feat_a)
                        if feat_b in df_all.columns:
                            source_set.add(feat_b)
                        break  # un seul séparateur par nom d'interaction

            source_list = [f for f in source_set if f in df_all.columns]
            if not source_list:
                print(f"[WARN] {regime}: impossible de reconstruire les sources. "
                      f"TRAIN_START fallback = {TRAIN_START}")
                df_for_start = df_all[base_in_df] if base_in_df else pd.DataFrame(index=df_all.index)
            else:
                df_all_interactions = generate_all_interactions(
                    df_all[source_list],
                    base_features=source_list,
                    top_n=len(source_list),
                    rolling_w=INTERACTION_ROLLING_WINDOW
                )
                # Garder uniquement les colonnes d'interaction dont on a besoin
                available_inter = [f for f in inter_cols if f in df_all_interactions.columns]
                missing_inter   = [f for f in inter_cols if f not in df_all_interactions.columns]
                if missing_inter:
                    print(f"[WARN] {len(missing_inter)} interactions introuvables "
                          f"lors de la reconstruction TRAIN_START. Ignorées.")
                df_for_start = pd.concat(
                    [df_all[base_in_df], df_all_interactions[available_inter]],
                    axis=1
                )
        else:
            df_for_start = df_all[base_in_df]

        # Première date où TOUTES les features finales sont non-NaN
        first_valid = df_for_start.dropna(how="any").index.min()
        if pd.isna(first_valid):
            first_valid = pd.Timestamp(TRAIN_START)
        train_start_auto = first_valid.strftime("%Y-%m-%d")
        print(f"[TRAIN_START] {regime} h={horizon_days}j: {train_start_auto} "
              f"(déterminé automatiquement par couverture des features)")

        result[regime] = {
            "features":     top_final,
            "train_start":  train_start_auto,
            "n_interactions": n_interactions_final,
            "shap_top_base":  top_base,
            "horizon_days":   horizon_days,
        }

    return result


# ─── Exécution Phase 1 pour les 3 horizons ────────────────────────────────────
phase1_results = {}
for h in HORIZONS_TO_TEST:
    phase1_results[h] = run_phase1_shap(h, df_post_features, features_post_engineering)

# Sauvegarde Phase 1
p1_rows = []
for h, regimes in phase1_results.items():
    for regime, info in regimes.items():
        p1_rows.append({
            "Horizon_Days": h,
            "VIX_Regime": regime,
            "Train_Start_Auto": info["train_start"],
            "N_Features_Final": len(info["features"]),
            "N_Interactions": info["n_interactions"],
            "N_Base": len(info["features"]) - info["n_interactions"],
            "Features": json.dumps(info["features"]),
        })
p1_df = pd.DataFrame(p1_rows)
p1_df.to_csv(OUTPUT_DIR / "phase1_shap_features.csv", index=False)
print(f"\n[SAVE] phase1_shap_features.csv ({len(p1_df)} lignes)")
display(p1_df[["Horizon_Days","VIX_Regime","Train_Start_Auto","N_Features_Final","N_Interactions","N_Base"]])



################################################################################
# PHASE 1 SHAP — HORIZON = 1j
################################################################################

--- PHASE 1 | h=1j | CALM ---
[TARGET] VIX column: VIX_Price  |  mode=fixed
[TARGET] VIX regime thresholds: CALM < 14.81, NORMAL [14.81-21.10), STRESS >= 21.10  (fixe, calculé sur train)
[TARGET] VIX flat threshold: ±0.00%
[TARGET] Flat days removed completely: 256/6733
[TARGET] Remaining non-flat rows: 6477
VIX_Regime
NORMAL    2369
STRESS    2115
CALM      1993
Name: count, dtype: int64
[SHAP CALM base] Top-40 sélectionnées sur 985 candidates
[INTERACTIONS] 20 features de base → 190 paires × 5 opérations = jusqu'à 950 colonnes
[INTERACTIONS] 1140 colonnes générées (après suppression des colonnes entièrement NaN)
[SHAP CALM base+interactions] Top-30 sélectionnées sur 1180 candidates
[SHAP FINAL] CALM h=1j: 30 features retenues (25 interactions, 5 base)
[TARGET] VIX column: VIX_Price  |  mode=fi

,Horizon_Days,VIX_Regime,Train_Start_Auto,N_Features_Final,N_Interactions,N_Base
0,1,CALM,2001-08-14,30,25,5
1,1,NORMAL,2000-11-02,30,26,4
2,1,STRESS,2000-11-17,30,26,4
3,1,GLOBAL,2000-11-15,30,20,10
4,3,CALM,2001-08-14,30,21,9
5,3,NORMAL,2000-11-02,30,26,4
6,3,STRESS,2000-11-15,30,29,1
7,3,GLOBAL,2000-11-15,30,24,6
8,5,CALM,2001-08-01,30,21,9
9,5,NORMAL,2000-11-15,30,24,6


In [49]:
# =============================================================================
# PHASE 2 : entraînement final — split fixe train < 2022 / test 2022→2026
# =============================================================================
# Pour chaque (horizon, régime) :
#   - Feature set et TRAIN_START issus de Phase 1 SHAP
#   - Reconstruction des interactions sur la fenêtre [TRAIN_START, fin]
#   - Split fixe : train < 2022, test 2022→2026
#   - 5 algos × N=5-15 features (ordre SHAP conservé)
#   - SMOTE systématique sur tous les régimes
# =============================================================================

from sklearn.model_selection import GridSearchCV

print("="*80)
print("PHASE 2 : entraînement final (split fixe 2022)")
print("="*80)

final_rows = []
model_number = 0

for h in HORIZONS_TO_TEST:
    for regime in ALL_REGIMES:
        info = phase1_results[h].get(regime)
        if info is None:
            print(f"[WARN] h={h} {regime}: pas de résultat Phase 1. Skip.")
            continue

        top_final   = info["features"]
        train_start = info["train_start"]

        print(f"\n{'='*60}")
        print(f"h={h}j | {regime} | TRAIN_START={train_start} | {len(top_final)} features")
        print(f"{'='*60}")

        # ── Reconstruire le dataset complet sur [train_start, fin] ──────────
        df_window = df_post_features.loc[
            df_post_features.index >= pd.Timestamp(train_start)
        ].copy()

        tb = TargetBuilder(q_low=0.33, q_high=0.67, mode="fixed",
                           rolling_window=ROLLING_QUANTILE_WINDOW,
                           horizon_days=h)
        df_full = tb.build(df_window, train_end=TEST_DATE)

        # Reconstruire les interactions sur df_full
        base_feats_in_df  = [f for f in top_final if f in df_full.columns]
        inter_feats_needed = [f for f in top_final if f not in df_full.columns]

        if inter_feats_needed:
            source_set = set(base_feats_in_df)
            # Ajouter les features source des interactions si nécessaire
            for fname in inter_feats_needed:
                parts = fname.split("__")
                if len(parts) >= 3:
                    for candidate in parts:
                        if candidate in df_full.columns:
                            source_set.add(candidate)
            df_inter = generate_all_interactions(
                df_full[list(source_set)],
                base_features=list(source_set),
                top_n=len(source_set),
                rolling_w=INTERACTION_ROLLING_WINDOW
            )
            available_inter = [f for f in inter_feats_needed if f in df_inter.columns]
            missing_inter   = [f for f in inter_feats_needed if f not in df_inter.columns]
            if missing_inter:
                print(f"[WARN] {len(missing_inter)} features d'interaction introuvables après reconstruction. Ignorées.")
            df_full = pd.concat([df_full, df_inter[available_inter]], axis=1)

        # Features finales disponibles (dans l'ordre SHAP)
        final_feats_available = [f for f in top_final if f in df_full.columns]
        if len(final_feats_available) < MIN_N_FEATURES:
            print(f"[WARN] Seulement {len(final_feats_available)} features disponibles. Skip.")
            continue

        df_full[final_feats_available] = df_full[final_feats_available].replace(
            [np.inf, -np.inf], np.nan
        )

        # ── Split fixe ──────────────────────────────────────────────────────
        df_train = df_full.loc[df_full.index < pd.Timestamp(TEST_DATE)].copy()
        df_test  = df_full.loc[df_full.index >= pd.Timestamp(TEST_DATE)].copy()

        if regime == "GLOBAL":
            train_mask = pd.Series(True, index=df_train.index)
            test_mask  = pd.Series(True, index=df_test.index)
        else:
            train_mask = df_train["VIX_Regime"] == regime
            test_mask  = df_test["VIX_Regime"]  == regime

        df_tr = df_train.loc[train_mask]
        df_te = df_test.loc[test_mask]

        if len(df_tr) < 50 or len(df_te) < 10:
            print(f"[WARN] train={len(df_tr)}, test={len(df_te)} — trop petit. Skip.")
            continue

        y_train_raw = df_tr["VIX_Direction"].values
        y_test      = df_te["VIX_Direction"].values

        if len(np.unique(y_train_raw)) < 2 or len(np.unique(y_test)) < 2:
            print(f"[WARN] Une seule classe présente. Skip.")
            continue

        cleaner = TrainFittedCleaner()
        X_tr_clean = cleaner.fit_transform(df_tr[final_feats_available])
        X_te_clean = cleaner.transform(df_te[final_feats_available])

        scaler = StandardScaler()
        X_tr_sc = pd.DataFrame(scaler.fit_transform(X_tr_clean),
                                columns=final_feats_available, index=df_tr.index)
        X_te_sc = pd.DataFrame(scaler.transform(X_te_clean),
                                columns=final_feats_available, index=df_te.index)

        print(f"  Train: {df_tr.index.min().date()} → {df_tr.index.max().date()} "
              f"(n={len(df_tr)}, UP={y_train_raw.sum()}, DOWN={len(y_train_raw)-y_train_raw.sum()})")
        print(f"  Test : {df_te.index.min().date()} → {df_te.index.max().date()} "
              f"(n={len(df_te)}, UP={y_test.sum()}, DOWN={len(y_test)-y_test.sum()})")

        configs = model_configs()
        max_n = min(MAX_N_FEATURES, len(final_feats_available))

        for algo_name in ALL_ALGOS:
            Model_class, param_grid, fixed_args = configs[algo_name]

            for n in range(MIN_N_FEATURES, max_n + 1):
                feats_n = final_feats_available[:n]

                X_tr_n = X_tr_sc[feats_n].copy()
                X_te_n = X_te_sc[feats_n].copy()

                # SMOTE systématique
                sm = SMOTE(random_state=RANDOM_STATE)
                uniq, counts = np.unique(y_train_raw, return_counts=True)
                if len(uniq) > 1 and min(counts) > 1:
                    X_tr_arr, y_tr = sm.fit_resample(X_tr_n.values, y_train_raw)
                    X_tr_n = pd.DataFrame(X_tr_arr, columns=feats_n)
                else:
                    y_tr = y_train_raw

                cv = TimeSeriesSplit(n_splits=3)
                try:
                    from sklearn.model_selection import GridSearchCV as GS
                    grid = GS(Model_class(**fixed_args), param_grid=param_grid,
                              scoring="roc_auc", cv=cv, n_jobs=-1)
                    grid.fit(X_tr_n.values, y_tr)
                    model = grid.best_estimator_
                except Exception as e:
                    continue

                y_pred  = model.predict(X_te_n.values)
                y_proba = (model.predict_proba(X_te_n.values)[:,1]
                           if hasattr(model, "predict_proba") else y_pred.astype(float))
                metrics = compute_metrics(y_test, y_pred, y_proba)

                model_number += 1
                row = {
                    "Model_Number": model_number,
                    "Horizon_Days": h,
                    "Model": algo_name,
                    "VIX_Regime": regime,
                    "Train_Start_Used": train_start,
                    "N_Features": n,
                    "Features": json.dumps(feats_n),
                    "SMOTE_Used": True,
                }
                row.update(metrics)
                final_rows.append(row)

            print(f"  [{algo_name}] N={MIN_N_FEATURES}..{max_n} testés.")

results_df = pd.DataFrame(final_rows)
results_df.to_csv(OUTPUT_DIR / "phase2_all_results.csv", index=False)
print(f"\n[DONE] {len(results_df)} lignes générées.")


PHASE 2 : entraînement final (split fixe 2022)

h=1j | CALM | TRAIN_START=2001-08-14 | 30 features
[TARGET] VIX column: VIX_Price  |  mode=fixed
[TARGET] VIX regime thresholds: CALM < 14.62, NORMAL [14.62-20.61), STRESS >= 20.61  (fixe, calculé sur train)
[TARGET] VIX flat threshold: ±0.00%
[TARGET] Flat days removed completely: 247/6472
[TARGET] Remaining non-flat rows: 6225
VIX_Regime
NORMAL    2282
STRESS    2049
CALM      1894
Name: count, dtype: int64
[INTERACTIONS] 23 features de base → 253 paires × 5 opérations = jusqu'à 1265 colonnes
[INTERACTIONS] 1518 colonnes générées (après suppression des colonnes entièrement NaN)
[WARN] 10 features d'interaction introuvables après reconstruction. Ignorées.
  Train: 2004-01-21 → 2020-02-19 (n=1676, UP=842, DOWN=834)
  Test : 2023-06-02 → 2026-01-09 (n=218, UP=113, DOWN=105)
  [XGBoost] N=5..15 testés.
  [LightGBM] N=5..15 testés.
  [GradientBoosting] N=5..15 testés.
  [RandomForest] N=5..15 testés.
  [LogisticRegression] N=5..15 testés.

h

In [50]:
# =============================================================================
# SYNTHÈSE ET EXPORT
# =============================================================================

REFS = {
    "CALM":  {"F1":0.667,"AUC":0.615,"Prec":0.627,"Recall":0.712},
    "NORMAL":{"F1":0.564,"AUC":0.608,"Prec":0.591,"Recall":0.540},
    "STRESS":{"F1":0.470,"AUC":0.613,"Prec":0.429,"Recall":0.519},
    "GLOBAL":{"F1":0.545,"AUC":0.604,"Prec":0.535,"Recall":0.556},
}

print("="*80)
print("MEILLEURE CONFIG PAR (HORIZON, RÉGIME) vs RÉFÉRENCES")
print("="*80)

best_rows = []
for h in HORIZONS_TO_TEST:
    for regime in ALL_REGIMES:
        sub = results_df[(results_df["Horizon_Days"]==h) & (results_df["VIX_Regime"]==regime)]
        if sub.empty:
            continue
        ref = REFS[regime]
        best = sub.loc[sub["AUC"].idxmax()]
        f1_gap  = best["F1"]  - ref["F1"]
        auc_gap = best["AUC"] - ref["AUC"]

        if f1_gap > 0.01 and auc_gap > 0.005:
            verdict = "✓ AMELIORATION"
        elif f1_gap < -0.02 or auc_gap < -0.01:
            verdict = "✗ REGRESSION"
        else:
            verdict = "≈ STABLE"

        print(f"\n[h={h}j | {regime}]  {verdict}")
        print(f"  Modèle : {best['Model']} N={best['N_Features']} | Train_Start={best['Train_Start_Used']}")
        print(f"  AUC={best['AUC']:.4f} (ref={ref['AUC']}, gap={auc_gap:+.4f})")
        print(f"  F1 ={best['F1']:.4f} (ref={ref['F1']}, gap={f1_gap:+.4f})")
        print(f"  P/R={best['Precision']:.3f}/{best['Recall']:.3f} (ref={ref['Prec']}/{ref['Recall']})")

        best_rows.append({
            "Horizon_Days":h, "VIX_Regime":regime,
            "Model":best["Model"],"N_Features":int(best["N_Features"]),
            "Train_Start_Used":best["Train_Start_Used"],
            "AUC":round(best["AUC"],4),"F1":round(best["F1"],4),
            "Precision":round(best["Precision"],3),"Recall":round(best["Recall"],3),
            "AUC_ref":ref["AUC"],"F1_ref":ref["F1"],
            "AUC_gap":round(auc_gap,4),"F1_gap":round(f1_gap,4),
            "Verdict":verdict,
        })

best_df = pd.DataFrame(best_rows)

excel_path = OUTPUT_DIR / "vix_shap_interactions_final_report.xlsx"
with pd.ExcelWriter(excel_path, engine="xlsxwriter") as writer:
    p1_df.to_excel(writer, sheet_name="Phase1_SHAP_Features", index=False)
    results_df.to_excel(writer, sheet_name="Phase2_All_Results", index=False)
    best_df.to_excel(writer, sheet_name="Best_Per_Horizon_Regime", index=False)

print(f"\n[SAVE] {excel_path}")
display(best_df)


MEILLEURE CONFIG PAR (HORIZON, RÉGIME) vs RÉFÉRENCES

[h=1j | CALM]  ✗ REGRESSION
  Modèle : RandomForest N=12 | Train_Start=2001-08-14
  AUC=0.6406 (ref=0.615, gap=+0.0256)
  F1 =0.6333 (ref=0.667, gap=-0.0337)
  P/R=0.598/0.673 (ref=0.627/0.712)

[h=1j | NORMAL]  ✗ REGRESSION
  Modèle : XGBoost N=8 | Train_Start=2000-11-02
  AUC=0.5373 (ref=0.608, gap=-0.0707)
  F1 =0.5143 (ref=0.564, gap=-0.0497)
  P/R=0.479/0.556 (ref=0.591/0.54)

[h=1j | STRESS]  ✗ REGRESSION
  Modèle : GradientBoosting N=15 | Train_Start=2000-11-17
  AUC=0.5565 (ref=0.613, gap=-0.0565)
  F1 =0.4269 (ref=0.47, gap=-0.0431)
  P/R=0.432/0.422 (ref=0.429/0.519)

[h=1j | GLOBAL]  ✗ REGRESSION
  Modèle : LogisticRegression N=7 | Train_Start=2000-11-15
  AUC=0.5863 (ref=0.604, gap=-0.0177)
  F1 =0.5534 (ref=0.545, gap=+0.0084)
  P/R=0.502/0.616 (ref=0.535/0.556)

[h=3j | CALM]  ✗ REGRESSION
  Modèle : LogisticRegression N=5 | Train_Start=2001-08-14
  AUC=0.6267 (ref=0.615, gap=+0.0117)
  F1 =0.5636 (ref=0.667, gap=-0.10

,Horizon_Days,VIX_Regime,Model,N_Features,Train_Start_Used,AUC,F1,Precision,Recall,AUC_ref,F1_ref,AUC_gap,F1_gap,Verdict
0,1,CALM,RandomForest,12,2001-08-14,0.6406,0.6333,0.598,0.673,0.615,0.667,0.0256,-0.0337,✗ REGRESSION
1,1,NORMAL,XGBoost,8,2000-11-02,0.5373,0.5143,0.479,0.556,0.608,0.564,-0.0707,-0.0497,✗ REGRESSION
2,1,STRESS,GradientBoosting,15,2000-11-17,0.5565,0.4269,0.432,0.422,0.613,0.470,-0.0565,-0.0431,✗ REGRESSION
3,1,GLOBAL,LogisticRegression,7,2000-11-15,0.5863,0.5534,0.502,0.616,0.604,0.545,-0.0177,0.0084,✗ REGRESSION
4,3,CALM,LogisticRegression,5,2001-08-14,0.6267,0.5636,0.646,0.500,0.615,0.667,0.0117,-0.1034,✗ REGRESSION
5,3,NORMAL,LogisticRegression,15,2000-11-02,0.5692,0.5842,0.540,0.637,0.608,0.564,-0.0388,0.0202,✗ REGRESSION
6,3,STRESS,GradientBoosting,6,2000-11-15,0.5573,0.4737,0.404,0.571,0.613,0.470,-0.0557,0.0037,✗ REGRESSION
7,3,GLOBAL,RandomForest,14,2000-11-15,0.6180,0.5536,0.569,0.539,0.604,0.545,0.0140,0.0086,≈ STABLE
8,5,CALM,LogisticRegression,9,2001-08-01,0.6717,0.5740,0.762,0.460,0.615,0.667,0.0567,-0.0930,✗ REGRESSION
9,5,NORMAL,XGBoost,14,2000-11-15,0.5793,0.5967,0.563,0.635,0.608,0.564,-0.0287,0.0327,✗ REGRESSION
